In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sqlalchemy import text
from src import config as C
from src.db_utils import run_sql_file
from src.simulation import simulate_series

engine = C.get_engine()
params = pd.read_sql(text("SELECT * FROM policy_params"), engine)
for c in ["unit_price", "unit_cost", "holding_cost_rate", "order_cost", "lead_time_days"]:
    params[c] = params[c].astype(float)

sales = pd.read_sql(text("SELECT date, store, item, units FROM sales WHERE date BETWEEN :a AND :b"),
                    engine, params={"a": C.SIM_START, "b": C.SIM_END}, parse_dates=["date"])
fc = pd.read_sql(text("""SELECT model, store, item, week_start, forecast FROM fact_forecast
                         WHERE model IN ('naive', 'lightgbm')"""), engine, parse_dates=["week_start"])

demand = {k: g.sort_values("date")["units"].to_numpy(float)
          for k, g in sales.groupby(["store", "item"])}
fcs = {(m, s, i): g.sort_values("week_start")["forecast"].to_numpy(float)
       for (m, s, i), g in fc.groupby(["model", "store", "item"])}

print("daily lengths:", {len(v) for v in demand.values()})
print("weekly forecast lengths:", {len(v) for v in fcs.values()})

daily lengths: {364}
weekly forecast lengths: {52}


In [2]:
POLICIES = {"naive": ("naive", False),
            "forecast_no_ss": ("lightgbm", False),
            "forecast_ss": ("lightgbm", True)}

rows = []
for r in params.itertuples():
    for pol, (model, use_ss) in POLICIES.items():
        k = simulate_series(demand[(r.store, r.item)], fcs[(model, r.store, r.item)],
                            r.safety_stock if use_ss else 0.0, r.eoq, r.lead_time_days,
                            r.unit_cost, r.unit_price, r.holding_cost_rate, r.order_cost,
                            C.STOCKOUT_PENALTY_MULT)
        rows.append({"store": r.store, "item": r.item, "policy": pol, **k})

sim = pd.DataFrame(rows)
print(sim.shape)

(1500, 12)


In [3]:
run_sql_file("07_sim_results.sql")
sim.to_sql("sim_results", engine, if_exists="append", index=False, chunksize=500, method="multi")

summary = sim.groupby("policy").agg(
    demand=("demand_units", "sum"), unmet=("unmet_units", "sum"),
    stockout_days=("stockouts", "sum"), avg_inventory=("avg_inventory", "sum"),
    holding=("holding_cost", "sum"), ordering=("ordering_cost", "sum"),
    stockout_cost=("stockout_cost", "sum"), total=("total_cost", "sum"))
summary["fill_rate"] = 1 - summary["unmet"] / summary["demand"]   # demand-weighted, not a mean of ratios
summary.to_csv(C.OUT_DIR / "07_sim_summary.csv")
print(summary[["fill_rate", "stockout_days", "avg_inventory", "holding",
               "ordering", "stockout_cost", "total"]].round(3))

                fill_rate  stockout_days  avg_inventory     holding  \
policy                                                                
forecast_no_ss      0.997           1522     393455.625  235027.973   
forecast_ss         1.000            189     412290.384  247959.168   
naive               0.994           2292     393756.146  235186.199   

                 ordering  stockout_cost       total  
policy                                                
forecast_no_ss  233997.82      92648.921  561674.714  
forecast_ss     235606.04       8863.787  492428.995  
naive           233229.09     180590.936  649006.225  


In [4]:
print(summary[["demand", "unmet"]].round(1))

                    demand    unmet
policy                             
forecast_no_ss  10710031.0  34757.9
forecast_ss     10710031.0   3223.9
naive           10710031.0  66474.9
